# Sesi 6 — Knowledge ERP: Prompting ReAct

Bagian dari kurikulum **Agentic Knowledge System — FastAPI + DuckDB + Qwen (Local, tanpa LLM cloud)**.

> Jalankan sel dari atas ke bawah secara berurutan. Sel bertanda `🔨 TODO` wajib dilengkapi peserta; solusi contoh disediakan pada sel setelahnya (bisa disembunyikan saat sesi diberikan ke peserta).

## 0. Setup Environment

In [ ]:
!pip -q install llama-cpp-python duckdb fastapi httpx

## 1. Model Qwen Lokal (sama seperti Sesi 3)

In [ ]:
import os

MODEL_PATH = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf"

from llama_cpp import Llama
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_threads=4, verbose=False)

def qwen_llm_call(prompt, stop=None):
    out = llm(prompt, max_tokens=256, stop=stop or ["Observation:"])
    return out["choices"][0]["text"]

## 2. Siapkan Ulang ERP API (dari Sesi 5, ringkas)

In [ ]:
import duckdb, uuid, json
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

erp_con = duckdb.connect("erp.duckdb")
erp_con.execute("""
CREATE TABLE IF NOT EXISTS products (id VARCHAR PRIMARY KEY, name VARCHAR, price DOUBLE, stock INTEGER);
CREATE TABLE IF NOT EXISTS customers (id VARCHAR PRIMARY KEY, name VARCHAR, email VARCHAR);
CREATE TABLE IF NOT EXISTS orders (id VARCHAR PRIMARY KEY, customer_id VARCHAR, status VARCHAR, created_at TIMESTAMP DEFAULT current_timestamp);
CREATE TABLE IF NOT EXISTS order_items (id VARCHAR PRIMARY KEY, order_id VARCHAR, product_id VARCHAR, qty INTEGER, price DOUBLE);
""")
if erp_con.execute("SELECT COUNT(*) FROM products").fetchone()[0] == 0:
    for name, price, stock in [("Kaos Polos Hitam", 75000, 50), ("Jaket Hoodie", 250000, 15)]:
        erp_con.execute("INSERT INTO products VALUES (?, ?, ?, ?)", [str(uuid.uuid4()), name, price, stock])
if erp_con.execute("SELECT COUNT(*) FROM customers").fetchone()[0] == 0:
    erp_con.execute("INSERT INTO customers VALUES (?, ?, ?)", [str(uuid.uuid4()), "Budi Santoso", "budi@example.com"])

erp_app = FastAPI()

@erp_app.get("/products")
def list_products():
    rows = erp_con.execute("SELECT id, name, price, stock FROM products").fetchall()
    return [{"id": r[0], "name": r[1], "price": r[2], "stock": r[3]} for r in rows]

@erp_app.post("/orders")
def create_order(payload: dict):
    customer_id = payload["customer_id"]
    items = payload["items"]
    for item in items:
        stock = erp_con.execute("SELECT stock FROM products WHERE id=?", [item["product_id"]]).fetchone()
        if not stock or stock[0] < item["qty"]:
            raise HTTPException(400, "Stok tidak cukup")
    order_id = str(uuid.uuid4())
    erp_con.execute("INSERT INTO orders VALUES (?, ?, 'pending', current_timestamp)", [order_id, customer_id])
    for item in items:
        price = erp_con.execute("SELECT price FROM products WHERE id=?", [item["product_id"]]).fetchone()[0]
        erp_con.execute("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)",
                        [str(uuid.uuid4()), order_id, item["product_id"], item["qty"], price])
        erp_con.execute("UPDATE products SET stock = stock - ? WHERE id=?", [item["qty"], item["product_id"]])
    return {"order_id": order_id, "status": "pending"}

@erp_app.get("/orders/{order_id}")
def get_order(order_id: str):
    order = erp_con.execute("SELECT id, customer_id, status FROM orders WHERE id=?", [order_id]).fetchone()
    if not order:
        raise HTTPException(404, "Order not found")
    return {"id": order[0], "customer_id": order[1], "status": order[2]}

erp_client = TestClient(erp_app)

## 3. Prompt ReAct Khusus ERP (dengan Guardrail)
Karena agent ini bisa **menulis data** (buat order), kita wajibkan `check_stock` sebelum `create_order`, dan `Action Input` berformat JSON agar parsing lebih andal.

In [ ]:
ERP_REACT_SYSTEM = """Kamu adalah agent ERP. Tool tersedia:
- check_stock[{"product_name": str}]: cek stok produk berdasarkan nama (partial match).
- create_order[{"customer_id": str, "product_id": str, "qty": int}]: buat order baru.
- get_order_status[{"order_id": str}]: cek status order.

PENTING: sebelum memanggil create_order, panggil check_stock dulu untuk memastikan stok cukup.

Format:
Thought: <pemikiranmu>
Action: <nama_tool>
Action Input: <JSON parameter>

Jika selesai:
Thought: <pemikiran akhir>
Final Answer: <jawaban ke user>
"""

## 4. Tool Executor ERP

In [ ]:
def call_erp_tool(action, params: dict):
    if action == "check_stock":
        products = erp_client.get("/products").json()
        return [p for p in products if params["product_name"].lower() in p["name"].lower()]
    if action == "create_order":
        payload = {"customer_id": params["customer_id"],
                   "items": [{"product_id": params["product_id"], "qty": params["qty"]}]}
        return erp_client.post("/orders", json=payload).json()
    if action == "get_order_status":
        return erp_client.get(f"/orders/{params['order_id']}").json()
    return {"error": "unknown tool"}

## 5. ReAct Loop ERP (dengan Human-in-the-loop Konfirmasi)

In [ ]:
import re

def erp_react_loop(query, max_steps=5, require_confirmation=True, verbose=True):
    history = f"{ERP_REACT_SYSTEM}\nQuestion: {query}\n"
    for step in range(max_steps):
        output = qwen_llm_call(history)
        if verbose:
            print(f"--- Step {step} ---\n{output}\n")
        if "Final Answer:" in output:
            return output.split("Final Answer:")[-1].strip()

        action_match = re.search(r"Action:\s*(\w+)", output)
        input_match = re.search(r"Action Input:\s*(\{.*\})", output, re.DOTALL)
        if not action_match or not input_match:
            history += output + "\nThought: Format salah, saya ulangi.\n"
            continue

        action = action_match.group(1).strip()
        try:
            params = json.loads(input_match.group(1))
        except json.JSONDecodeError:
            history += output + "\nObservation: Parameter JSON tidak valid.\n"
            continue

        if action == "create_order" and require_confirmation:
            confirm = input(f"Agent ingin membuat order: {params}. Konfirmasi? (y/n): ")
            if confirm.lower() != "y":
                history += output + "\nObservation: Order dibatalkan oleh user.\n"
                continue

        observation = call_erp_tool(action, params)
        history += output + f"\nObservation: {observation}\n"
    return "Maaf, saya belum menyelesaikan permintaan dalam batas langkah."

### 🔨 TODO 1
Tambahkan tabel log `agent_logs (id, query, action, params, observation, created_at)` di `erp.duckdb`, lalu catat setiap `Action`+`Observation` pada `erp_react_loop` untuk audit trail.

In [ ]:
# TODO: buat tabel agent_logs dan catat setiap langkah
erp_con.execute("""
CREATE TABLE IF NOT EXISTS agent_logs (
    id VARCHAR PRIMARY KEY, query VARCHAR, action VARCHAR,
    params VARCHAR, observation VARCHAR, created_at TIMESTAMP DEFAULT current_timestamp
);
""")
# TODO: panggil erp_con.execute("INSERT INTO agent_logs VALUES (...)") di dalam erp_react_loop

### 🔨 TODO 2
Uji `erp_react_loop` dengan permintaan yang produknya sudah habis stok. Pastikan agent menolak dengan sopan, bukan tetap membuat order.

In [ ]:
# TODO: uji kasus stok habis
# result = erp_react_loop("Pesankan 100 Jaket Hoodie untuk Budi", require_confirmation=False, verbose=False)
# print(result)

## Ringkasan Sesi 6
Agent ERP berbasis Qwen kini bisa mengecek stok dan membuat order lewat bahasa natural, dengan guardrail konfirmasi. Sesi 7 memakai Qwen (model yang sama) untuk **generate laporan**, bukan sekadar aksi.